# block-group-stack — ex2: introspect a BlockGroup and verify the shape-change invariant

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `block-group-stack`. Running the final beacon cell reports progress against the `CNN: BlockGroup stack` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: BlockGroup stack` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`block-group-stack`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "block-group-stack"
DD_SUBTOPIC = "CNN: BlockGroup stack"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Introspecting a BlockGroup's shape-change pattern — quick refresher

A correctly-built ResNet BlockGroup has exactly **ONE** shape-changing block — block 0 — followed by `n_blocks - 1` identity-shaped blocks (in_feats == out_feats AND first_stride == 1). Verifying this invariant is the cleanest way to detect a mis-stacked group.

**The introspection idiom.** Given a `nn.Sequential` of `ResBlock`s, walk children and tabulate:

```
shape_changers = [
    i for i, b in enumerate(group)
    if b.in_feats != b.out_feats or b.first_stride != 1
]
```

Then assert `shape_changers == [0]` (only block 0 changes shape) AND `group[0].out_feats == group[-1].out_feats` (output width consistent).

**Why this matters.** If somebody slips a downsample into block 3, the residual skip-connection breaks (input/output shapes no longer match) and the model silently underperforms or crashes at forward time. The structural check catches it before any data flows.

**Generalization.** The same one-shape-change invariant holds for every BlockGroup variant — Bottleneck-ResNet, ResNeXt, DenseNet's transition blocks. As long as a 'group' is the unit of width/resolution change, exactly one block at the top of the group does the changing.

### Exercise 2 — introspect a BlockGroup and verify the shape-change invariant

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Evaluate
> LO: Evaluate a `nn.Sequential` of `ResBlock`s against the canonical BlockGroup invariant (exactly one shape-changing block at index 0, all subsequent blocks identity-shaped) by walking children and returning a structured audit.
> Keywords: resnet, block-group, introspection, invariant
> ```

**KCs targeted:** `block-group-one-shape-changer`, `block-group-output-width-consistency`

A toy `ResBlock` is provided (same as the ex1 drill):

```
class ResBlock(nn.Module):
    def __init__(self, in_feats, out_feats, first_stride=1):
        super().__init__()
        self.in_feats     = in_feats
        self.out_feats    = out_feats
        self.first_stride = first_stride
        self.proj = nn.Conv2d(in_feats, out_feats, kernel_size=1, stride=first_stride)
    def forward(self, x):
        return self.proj(x)
```

Implement `ex2_audit_block_group(group)`. Given a `nn.Sequential` of `ResBlock`s, return a dict:

```python
{
  'n_blocks':              int,                       # total blocks in the group
  'shape_changers':        List[int],                 # indices of blocks where in_feats != out_feats OR first_stride != 1
  'is_canonical':          bool,                      # True iff shape_changers == [0] (and group is non-empty)
  'output_width':          int,                       # group[-1].out_feats
  'output_width_consistent': bool,                    # True iff EVERY block has out_feats == output_width
}
```

**The canonical invariant** is: exactly ONE shape-changing block, and it's at index 0. Every subsequent block has `in_feats == out_feats` AND `first_stride == 1`. The introspection walks `group` (which iterates its children) and tabulates the facts.

**Edge cases.**
- `len(group) == 1` → `shape_changers` is `[0]` if that block is a shape-changer, else `[]`. `is_canonical` is True iff `shape_changers == [0]`.
- Empty `nn.Sequential` → `n_blocks == 0`, `shape_changers == []`, `is_canonical == False`, `output_width_consistent == True` (vacuous).

The test instantiates correctly-built groups, deliberately-miswired ones (downsample in the middle, mismatched widths), and confirms your audit catches each case.

In [ ]:
def ex2_audit_block_group(group) -> dict:
    blocks = list(group)
    n = len(blocks)
    shape_changers = [
        i for i, b in enumerate(blocks)
        if b.in_feats != b.out_feats or b.first_stride != 1
    ]
    if n == 0:
        return {
            'n_blocks': 0,
            'shape_changers': [],
            'is_canonical': False,
            'output_width': 0,
            'output_width_consistent': True,
        }
    output_width = blocks[-1].out_feats
    consistent = all(b.out_feats == output_width for b in blocks)
    return {
        'n_blocks': n,
        'shape_changers': shape_changers,
        'is_canonical': shape_changers == [0],
        'output_width': output_width,
        'output_width_consistent': consistent,
    }


<details><summary>Solution</summary>

```python
def ex2_audit_block_group(group) -> dict:
    blocks = list(group)
    n = len(blocks)
    shape_changers = [
        i for i, b in enumerate(blocks)
        if b.in_feats != b.out_feats or b.first_stride != 1
    ]
    if n == 0:
        return {
            'n_blocks': 0,
            'shape_changers': [],
            'is_canonical': False,
            'output_width': 0,
            'output_width_consistent': True,
        }
    output_width = blocks[-1].out_feats
    consistent = all(b.out_feats == output_width for b in blocks)
    return {
        'n_blocks': n,
        'shape_changers': shape_changers,
        'is_canonical': shape_changers == [0],
        'output_width': output_width,
        'output_width_consistent': consistent,
    }
```

**Why `is_canonical` requires `[0]` exactly.** Two failure modes both produce non-canonical groups:
1. `shape_changers == []` — no block changes shape; the group isn't doing anything structural.
2. `shape_changers == [0, 2, ...]` — multiple shape-changers; the group has been mis-stacked.

Both should fail the canonical check; the `== [0]` predicate captures both cases in one line.

**Why `output_width_consistent` is its own field.** A group can have `is_canonical == True` AND `output_width_consistent == True` simultaneously (the well-formed case), but they're *independently* useful diagnostics. Some bugs only break consistency (mismatched widths through the middle) without adding a second shape-changer.

**Why `list(group)` instead of `for b in group`.** `nn.Sequential` is iterable but its `__len__` is well-defined; `list(group)` makes the iteration and indexing explicit. For `enumerate(blocks)` and `blocks[-1]`, the list form is clearer than working with the underlying iterator.

**Empty-group edge case.** `n_blocks == 0` returns `is_canonical = False` (an empty group is trivially non-canonical — it can't have a shape-changer at index 0). `output_width_consistent = True` is vacuously true (no blocks to disagree). `output_width = 0` is a sentinel; the test doesn't rely on a specific value here, just that the call doesn't crash.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()